In [61]:
import pandas as pd
import numpy as np

In [62]:
violations = pd.read_csv("violations.csv")

In [63]:
count_viol = violations.groupby('sam_id').size()
count_viol = pd.DataFrame(count_viol).reset_index()
count_viol.columns = ['sam_id','no_of_violations']

In [64]:
count_viol = count_viol.sort_values(by='no_of_violations',ascending=False)

In [65]:
count_viol['bad_landlord'] = count_viol.apply(lambda x: 1 if x.iloc[1]>5 else 0,axis=1)

In [66]:
violations = pd.merge(violations,count_viol, on='sam_id', how='left')

### No. of bad landlords across boston

In [67]:
len(count_viol[count_viol['bad_landlord']==1][['sam_id','bad_landlord']].drop_duplicates())-1

195

### Combining this information with the Student addresses data

In [77]:
student_sam = pd.read_excel("Merged_Student_and_SAM.xlsx")

In [78]:
student_sam = pd.merge(student_sam, count_viol[['sam_id','no_of_violations', 'bad_landlord']],left_on='SAM_ADDRESS_ID',right_on='sam_id',how='left')

In [79]:
student_sam[student_sam['no_of_violations']>0]['sam_id'].drop_duplicates()

8          86421.0
9         114164.0
11        112521.0
23         36989.0
39           469.0
            ...   
321019     99093.0
321227     70909.0
321361     66066.0
321620     50934.0
322178     88860.0
Name: sam_id, Length: 3119, dtype: float64

### No of violations in student addresses

In [80]:
int(student_sam[['sam_id','no_of_violations']].drop_duplicates()['no_of_violations'].sum())

5086

### No of Bad Landlords of student occupied buildings

In [81]:
int(student_sam[['sam_id','bad_landlord']].drop_duplicates()['bad_landlord'].sum())

60

#### Adding standardized neighbourhoods

In [82]:
n_zip = pd.read_csv("Updated_Zip_Code_and_Neighborhood_Mapping.csv")
n_zip['Zip Code'] = n_zip['Zip Code'].astype(str)
n_zip['Zip Code'] = n_zip['Zip Code'].apply(lambda x: '0'+x) 

In [83]:
student_sam['ZIP_CODE'] = student_sam['ZIP_CODE'].astype(str)
student_sam['ZIP_CODE'] = student_sam['ZIP_CODE'].apply(lambda x: '0'+x[0:4]) 

In [84]:
student_sam['ZIP_CODE'].unique()

array(['02134', '02446', '02215', '02120', '0nan', '02135', '02125',
       '02127', '02115', '02124', '02116', '02119', '02128', '02130',
       '02131', '02136', '02118', '02121', '02122', '02126', '02129',
       '02132', '02199', '02111', '02109', '02113', '02210', '02114',
       '02108', '02110', '02163', '02467', '02186', '02458'], dtype=object)

In [85]:
student_sam = pd.merge(student_sam,n_zip,left_on='ZIP_CODE',right_on='Zip Code',how='left')

In [86]:
student_sam.columns

Index(['6a. street #', '6b. street name', '6c. street suffix', '6d. unit #',
       '6e. zip', '7. undergraduate (u) or graduate (g)',
       '8. full-time (ft) or part-time (pt)', '9. at-home or not-at-home',
       '9. 5 or more undergrads/unit (y/n)', 'university', 'year',
       'FULL_ADDRESS_Student', 'Latitude', 'Longitude', 'X', 'Y', 'OBJECTID',
       'SAM_ADDRESS_ID', 'BUILDING_ID', 'RELATIONSHIP_TYPE',
       'FULL_ADDRESS_SAM', 'STREET_NUMBER', 'IS_RANGE', 'RANGE_FROM',
       'RANGE_TO', 'UNIT', 'FULL_STREET_NAME', 'STREET_ID', 'STREET_PREFIX',
       'STREET_BODY', 'STREET_SUFFIX_ABBR', 'STREET_FULL_SUFFIX',
       'STREET_SUFFIX_DIR', 'STREET_NUMBER_SORT', 'MAILING_NEIGHBORHOOD',
       'ZIP_CODE', 'X_COORD', 'Y_COORD', 'SAM_STREET_ID', 'WARD',
       'PRECINCT_WARD', 'PARCEL', 'created_date', 'last_edited_date',
       'Unnamed: 44', 'Unnamed: 45', 'sam_id', 'no_of_violations',
       'bad_landlord', 'Zip Code', 'Neighbourhood'],
      dtype='object')

### Standardizing zip code in the violations dataset

In [87]:
violations['violation_zip'] = violations['violation_zip'].apply(lambda x: x[0:4] if len(str(x))>5 else x)
violations['violation_zip'] = violations['violation_zip'].apply(lambda x: np.nan if x==' ' else x)

In [88]:
violations['violation_zip'].unique()

array(['02128', '02122', '02125', '02121', '02136', '02131', '02132',
       '02126', '02124', '02134', '02114', '02119', '02111', '02108',
       '02116', '02127', '02135', '02118', '02215', '02130', nan, '02129',
       '02120', '02113', '02115', '02109', '02110', '02199', '02210',
       '0212', '02123', '02467', '02446'], dtype=object)

In [89]:
violations = pd.merge(violations,n_zip,left_on='violation_zip', right_on='Zip Code', how='left')

In [90]:
student_sam.to_csv('student_sam_neighbourhood_propertyviol_bl.csv')

In [91]:
student_sam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353897 entries, 0 to 353896
Data columns (total 51 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   6a. street #                          345222 non-null  object 
 1   6b. street name                       345466 non-null  object 
 2   6c. street suffix                     309157 non-null  object 
 3   6d. unit #                            221031 non-null  object 
 4   6e. zip                               333583 non-null  object 
 5   7. undergraduate (u) or graduate (g)  338592 non-null  object 
 6   8. full-time (ft) or part-time (pt)   353778 non-null  object 
 7   9. at-home or not-at-home             220501 non-null  object 
 8   9. 5 or more undergrads/unit (y/n)    89256 non-null   object 
 9   university                            353897 non-null  object 
 10  year                                  353897 non-null  object 
 11  

### Adding data about the building to the dataset

In [92]:
prop_ass = pd.read_csv("property_assessment.csv")

C:\Users\adity\AppData\Local\Temp\ipykernel_101292\2851409619.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  prop_ass = pd.read_csv("property_assessment.csv")


In [93]:
prop_ass.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182242 entries, 0 to 182241
Data columns (total 65 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   PID                  182242 non-null  int64  
 1   CM_ID                93291 non-null   float64
 2   GIS_ID               182242 non-null  int64  
 3   ST_NUM               172879 non-null  float64
 4   ST_NAME              182242 non-null  object 
 5   UNIT_NUM             82613 non-null   object 
 6   CITY                 182239 non-null  object 
 7   ZIP_CODE             182239 non-null  float64
 8   BLDG_SEQ             182242 non-null  int64  
 9   NUM_BLDGS            182242 non-null  int64  
 10  LUC                  182242 non-null  int64  
 11  LU                   182242 non-null  object 
 12  LU_DESC              182242 non-null  object 
 13  BLDG_TYPE            179626 non-null  object 
 14  OWN_OCC              182242 non-null  object 
 15  OWNER            

In [94]:
prop_ass_temp = prop_ass[['PID','OWN_OCC','OWNER','RES_UNITS','BLDG_VALUE']].copy(deep=True)

In [95]:
prop_ass_temp

,PID,OWN_OCC,OWNER,RES_UNITS,BLDG_VALUE
0,100001000,Y,PASCUCCI CARLO,NaN,"594,400"
1,100002000,N,SEMBRANO RODERICK,NaN,"619,700"
2,100003000,Y,GUERRA CHEVARRIA ANA S,NaN,"605,300"
3,100004000,N,JB REALTY TRUST,NaN,"535,600"
4,100005000,Y,MARKS TRAVIS JOSEPH,NaN,"501,400"
...,...,...,...,...,...
182237,2205666000,N,CITY OF BOSTON BY FCL,NaN,0
182238,2205667000,N,GREALISH MARTIN J TS,NaN,0
182239,2205668000,N,EAGLE PROPERTY HOLDINGS LLC,NaN,"850,500"
182240,2205669000,N,GREALISH MARTIN J TRST,NaN,"1,458,800"


##### Here PID is the Parcel ID

In [96]:
prop_ass_temp['BLDG_VALUE'] = prop_ass_temp['BLDG_VALUE'].str.replace(',', '')

In [97]:
prop_ass_temp['BLDG_VALUE'] = prop_ass_temp['BLDG_VALUE'].astype(int)

In [98]:
prop_ass_temp['BLDG_VALUE'].max()

np.int64(1928413000)

In [99]:
student_sam = pd.merge(student_sam,prop_ass_temp,left_on='PARCEL', right_on='PID', how='left')

In [100]:
student_sam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353897 entries, 0 to 353896
Data columns (total 56 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   6a. street #                          345222 non-null  object 
 1   6b. street name                       345466 non-null  object 
 2   6c. street suffix                     309157 non-null  object 
 3   6d. unit #                            221031 non-null  object 
 4   6e. zip                               333583 non-null  object 
 5   7. undergraduate (u) or graduate (g)  338592 non-null  object 
 6   8. full-time (ft) or part-time (pt)   353778 non-null  object 
 7   9. at-home or not-at-home             220501 non-null  object 
 8   9. 5 or more undergrads/unit (y/n)    89256 non-null   object 
 9   university                            353897 non-null  object 
 10  year                                  353897 non-null  object 
 11  

In [101]:
student_sam.to_csv('Final_dataset.csv', index=False)

In [102]:
student_sam

,6a. street #,6b. street name,6c. street suffix,6d. unit #,6e. zip,7. undergraduate (u) or graduate (g),8. full-time (ft) or part-time (pt),9. at-home or not-at-home,9. 5 or more undergrads/unit (y/n),university,...,sam_id,no_of_violations,bad_landlord,Zip Code,Neighbourhood,PID,OWN_OCC,OWNER,RES_UNITS,BLDG_VALUE
0,10,Higgins,ST,NaN,2134,U,FT,NaN,NaN,Emmanuel College,...,NaN,NaN,NaN,02134,Allston,NaN,NaN,NaN,NaN,NaN
1,10,Higgins,ST,NaN,2134,U,FT,NaN,NaN,Emmanuel College,...,NaN,NaN,NaN,02134,Allston,NaN,NaN,NaN,NaN,NaN
2,1189,Commonwealth,AVE,6,2134,U,FT,NaN,NaN,Emmanuel College,...,NaN,NaN,NaN,02134,Allston,2.100815e+09,N,PIZZUTI FAMILY INC,NaN,2332300.0
3,12,Glenville,AVE,NaN,2134,U,FT,NaN,NaN,Emmanuel College,...,NaN,NaN,NaN,02134,Allston,NaN,NaN,NaN,NaN,NaN
4,12,Glenville,AVE,1,2134,U,FT,NaN,NaN,Emmanuel College,...,NaN,NaN,NaN,02134,Allston,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353892,31,Robinwood,Ave,Apt #1,2130,NaN,FT,NaN,NaN,Northeastern Univerisity,...,469.0,1.0,0.0,02124,Dorchester,NaN,NaN,NaN,NaN,NaN
353893,61,S. Huntington 202,Ave,NaN,2130,NaN,FT,NaN,NaN,Northeastern Univerisity,...,469.0,1.0,0.0,02124,Dorchester,NaN,NaN,NaN,NaN,NaN
353894,29,Wellington,Street,#B04,2118,NaN,FT,NaN,NaN,Northeastern Univerisity,...,NaN,NaN,NaN,02118,South End,4.025460e+08,N,BURTON RONALD E JR TS,NaN,6124000.0
353895,1,Wigglesworth,St.,Apt 1,2120,NaN,FT,NaN,NaN,Northeastern Univerisity,...,NaN,NaN,NaN,02120,Mission Hill,1.000088e+09,N,WIGGLESWORTH INVESTMENTS LLC,NaN,859000.0


In [103]:
student_sam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353897 entries, 0 to 353896
Data columns (total 56 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   6a. street #                          345222 non-null  object 
 1   6b. street name                       345466 non-null  object 
 2   6c. street suffix                     309157 non-null  object 
 3   6d. unit #                            221031 non-null  object 
 4   6e. zip                               333583 non-null  object 
 5   7. undergraduate (u) or graduate (g)  338592 non-null  object 
 6   8. full-time (ft) or part-time (pt)   353778 non-null  object 
 7   9. at-home or not-at-home             220501 non-null  object 
 8   9. 5 or more undergrads/unit (y/n)    89256 non-null   object 
 9   university                            353897 non-null  object 
 10  year                                  353897 non-null  object 
 11  

### Exporting data to visualize violations data

In [107]:
violations.columns

Index(['case_no', 'ap_case_defn_key', 'status_dttm', 'status', 'code', 'value',
       'description', 'violation_stno', 'violation_sthigh', 'violation_street',
       'violation_suffix', 'violation_city', 'violation_state',
       'violation_zip', 'ward', 'contact_addr1', 'contact_addr2',
       'contact_city', 'contact_state', 'contact_zip', 'sam_id', 'latitude',
       'longitude', 'location', 'no_of_violations', 'bad_landlord', 'Zip Code',
       'Neighbourhood'],
      dtype='object')

In [108]:
violations['status_dttm'] = pd.to_datetime(violations['status_dttm'])

In [109]:
violations['year'] = violations['status_dttm'].dt.year

In [110]:
temp3 = violations.groupby(['Neighbourhood','year','description']).size()
temp3.to_csv('temp3.csv',index=False)

### Filtering the data to only include violations for student housing buildings

In [111]:
student_sam_list = list(student_sam['sam_id'].unique())

In [112]:
violations['student_housing'] = violations['sam_id'].apply(lambda x: 1 if x in student_sam_list else 0)

In [113]:
temp4 =  violations[violations['student_housing']==1].groupby(['Neighbourhood','year','description']).size()

In [114]:
temp4 = temp4.reset_index()
temp4.columns = ['Neighbourhood', 'year', 'description', 'count']
temp4.to_csv('temp4.csv',index=False)

In [115]:
violations.to_csv('updated_violations.csv',index=False)

[View the visualizations here](https://lookerstudio.google.com/reporting/04a5b8cc-d28a-4b61-b088-10d7c92ea456)